In [1]:
from src.utils import set_random_seed
import argparse
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn import MSELoss, BCEWithLogitsLoss
import numpy as np
import random
from src.data.featurizer import Vocab, N_ATOM_TYPES, N_BOND_TYPES
from src.data.finetune_dataset import MoleculeDataset,LatentDataset
from src.data.collator import Collator_tune,Collator_latent
from src.model.light import LiGhTPredictor as LiGhT
from src.trainer.scheduler import PolynomialDecayLR
from src.trainer.finetune_trainer import Trainer
from src.trainer.evaluator import Evaluator
from src.trainer.result_tracker import Result_Tracker
from src.model_config import config_dict

import warnings

In [2]:
def init_params(module):
    if isinstance(module, nn.Linear):
        module.weight.data.normal_(mean=0.0, std=0.02)
        if module.bias is not None:
            module.bias.data.zero_()
    if isinstance(module, nn.Embedding):
        module.weight.data.normal_(mean=0.0, std=0.02)
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
def parse_args():
    parser = argparse.ArgumentParser(description="Arguments for training LiGhT")
    parser.add_argument("--seed", type=int, default=22)
    parser.add_argument("--n_epochs", type=int, default=50)

    parser.add_argument("--config", type=str, default="base")
    parser.add_argument("--model_path", type=str, default="../models/pretrained/base/base.pth")
    parser.add_argument("--dataset", type=str,default="bace")
    parser.add_argument("--data_path", type=str,default="../datasets/")
    parser.add_argument("--dataset_type", type=str, default="classification")
    parser.add_argument("--metric", type=str, default="rocauc")
    parser.add_argument("--split", type=str, default="scaffold-0")

    parser.add_argument("--weight_decay", type=float, default=0.)
    parser.add_argument("--dropout", type=float, default=0.)
    parser.add_argument("--lr", type=float, default=3e-5)

    parser.add_argument("--n_threads", type=int, default=8)
    args = parser.parse_args([])
    return args

In [3]:
args = parse_args()

In [4]:
set_random_seed(args.seed)

In [5]:
config = config_dict[args.config]
vocab = Vocab(N_ATOM_TYPES, N_BOND_TYPES)

In [6]:
g = torch.Generator()
g.manual_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
collator = Collator_latent(config['path_length'])

In [7]:
dataset = LatentDataset(smiles_path=f"/home/user/qianyh/plifts_v2/data/external_test/smiles.pkl",
                        graph_path=f"./graphs_945.pkl",
                        fp_path=f"./rdkfp1-7_945.npz",
                        rd_path=f"./molecular_descriptorsbi_945.npz")

In [8]:
loader = DataLoader(dataset=dataset,batch_size=32,shuffle=False,num_workers=16,worker_init_fn=seed_worker, generator=g, drop_last=False, collate_fn=collator)

In [9]:
model = LiGhT(
        d_node_feats=config['d_node_feats'],
        d_edge_feats=config['d_edge_feats'],
        d_g_feats=config['d_g_feats'],
        d_fp_feats=512,
        d_md_feats=200,
        d_hpath_ratio=config['d_hpath_ratio'],
        n_mol_layers=config['n_mol_layers'],
        path_length=config['path_length'],
        n_heads=config['n_heads'],
        n_ffn_dense_layers=config['n_ffn_dense_layers'],
        input_drop=0,
        attn_drop=args.dropout,
        feat_drop=args.dropout,
        n_node_types=vocab.vocab_size
    ).to(device)

In [10]:
args.model_path = "./models/pretrained/base/base_old.pth"
model.load_state_dict({k.replace('module.',''):v for k,v in torch.load(args.model_path).items()})

<All keys matched successfully>

In [11]:
import os            
os.path.exists(args.model_path)

True

In [12]:
from tqdm import tqdm

In [13]:
model.load_state_dict({k.replace('module.',''):v for k,v in torch.load(args.model_path).items()})
from tqdm import tqdm
fps_list = []
for batch_idx, batched_data in enumerate(tqdm(loader)):
    (_, g, ecfp, md) = batched_data
    ecfp = ecfp.to(device)
    md = md.to(device)
    g = g.to(device)
    fps = model.generate_fps(g, ecfp, md)
    fps_list.extend(fps.detach().cpu().numpy().tolist())

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:02<00:00, 10.53it/s]


In [14]:
fps_emb = np.array(fps_list)

In [15]:
fps_emb.shape

(945, 2304)

In [17]:
import pickle  
import joblib
smiles = joblib.load("/home/user/qianyh/plifts_v2/data/external_test/smiles.pkl")
len(smiles)

945

In [18]:
kpgt_latent = {}

for i in range(len(smiles)):
    smi = smiles[i]
    emb = fps_emb[i]
    kpgt_latent[smi] = emb

In [19]:
kpgt_latent.keys()

dict_keys(['Clc1ccccc1OC1CCCc2ccc(N3CCNCC3)nc21', 'CC[C@H]1CCC[C@H](O[C@H]2CC[C@H](N(C)CCC[n+]3ccccc3)[C@@H](C)O2)[C@@H](C)C(=O)C2=C[C@@H]3[C@@H](C=C[C@@H]4C[C@@H](O[C@@H]5O[C@@H](C)[C@H](OC)[C@@H](OC)[C@H]5OC)C[C@@H]34)[C@@H]2CC(=O)O1', 'COC(=O)c1ccc2c(N(Cc3ccc(-c4ccc(OC)c(C)c4)c(C#N)c3)C(=O)CC(C)(C)C)nccc2c1', 'COc1ccc(CC(OC(=O)/C=C/c2ccccc2)C(=O)O)cc1OC', 'CC(=O)N1CCN(C(=O)c2ccc3oc(CCCc4ccccc4)nc3c2)CC1', 'CCN(C)C(C)C(O)c1ccccc1', 'CC(N)(CCc1ccc(OCCOc2cccc(F)c2F)cc1)COP(=O)(O)O', 'CCN(Cc1ccc(C(F)(F)F)cc1)c1ncnc(NCC2(O)CCN(CC(N)=O)CC2O)c1F', 'CC(O)(CCCCCCCCCS(=O)(=O)CCCC(F)(F)C(F)(F)F)C(=O)Nc1ccc(C#N)c(C(F)(F)F)c1', 'COC(=O)C=Cc1cccc(N(Cc2ccc(-c3ccc(N(C)C)c(C#N)c3)cc2)C(=O)C2CCCCC2)c1', 'FC(F)(F)c1ccc(CSc2nc3ccccc3o2)cc1', 'O=c1cc(N2CCOCC2)oc2cc(OCc3cccc[n+]3[O-])ccc12', 'COC(=O)C12OCC34C(CC5C(C)C(=O)C(=O)CC5(C)C3C(O)C1O)OC(=O)C(OC(=O)C=C(C)C(C)C)C24', 'N#Cc1ccccc1OCC(O)CNCCNC(=O)Nc1ccccc1', 'CC(Cn1ccc(-c2ccc(C#N)c(Cl)c2)n1)NC(=O)c1csc(N2CCOCC2)n1', 'CC1(C)C(=O)N(c2ccc(C#N)c(C(F)(F)F

In [20]:
import joblib

joblib.dump(kpgt_latent,"/home/user/qianyh/plifts_v2/data/external_test/d_graph_dict.pkl")

['/home/user/qianyh/plifts_v2/data/external_test/d_graph_dict.pkl']